In [ ]:
import json
import pydantic
import shapely
import geopandas as gpd
from websockets.sync.client import connect
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np

# Client library

In [ ]:
class Radar(pydantic.BaseModel):
    id: int
    lat: float
    lon: float
    power: int
    diameter: float
    frequency: float
    pulse_width: float
    cpi_pulses: int
    bandwidth: int
    pfa: float
    min_elevation: float
    max_elevation: float

In [ ]:
class OpenburstClient:
    _RADAR_COVERAGE: int = 559765
    _ELEVATION: int = 6200901

    def __init__(
        self, base_url: str, radterrain_port: int = 9978, geoplot_port: int = 9943
    ):
        self._url_radterrain = f"ws://{base_url}:{radterrain_port}/dem"
        self._url_geoplot = f"ws://{base_url}:{geoplot_port}/geoplot"

    def calculate_coverage(
        self,
        radar: Radar,
        target_flight_height: float,
        target_cross_section: float,
        enable_propagation_model: bool = False,
        enable_magl: bool = False,
    ) -> shapely.geometry.polygon.Polygon:
        """
        Calculate radar coverage.

        Returns
        -------
        Shapely polygon representing radar coverage in the WGS84 coordinate system.
        """
        # Build the request message.
        properties = [
            self._RADAR_COVERAGE,
            radar.id,
            radar.lat,
            radar.lon,
            target_flight_height,
            radar.power,
            radar.diameter,
            radar.frequency,
            radar.pulse_width,
            radar.cpi_pulses,
            radar.bandwidth,
            radar.pfa,
            target_cross_section,
            radar.min_elevation,
            radar.max_elevation,
            int(enable_propagation_model),
            int(enable_magl),
        ]

        # Calculate radar coverage.
        with connect(self._url_radterrain) as ws:
            ws.send(",".join([str(p) for p in properties]))
            answer = ws.recv()

        # Convert radar coverage to KML.
        with connect(self._url_geoplot) as ws:
            ws.send(
                json.dumps(
                    {
                        "request_type": "activeCoveragePoints",
                        "nbr_args": 1,
                        "args": ['"' + answer + '"'],
                    }
                )
            )
            answer2 = ws.recv()
            kml = json.loads(json.loads(answer2)["args"][0])

        # Parse KML file.
        with open("test.kml", "w") as file:
            file.write(kml)

        df = gpd.read_file("test.kml")
        polygon = df.iloc[0]["geometry"]

        return polygon

    def elevation_at(self, query_lat: float, query_lon: float) -> float:
        """Query elevation [m] at the query position."""
        # Needed for distance calculation of query point to POI. Dummy for elevation calc.
        poi_lat = 46.25
        poi_lon = 7.12

        query = [self._ELEVATION, query_lat, query_lon, poi_lat, poi_lon]
        query = [str(part) for part in query]

        with connect(self._url_radterrain) as ws:
            ws.send(",".join(query))
            answer = ws.recv()

        height = json.loads(answer)[1]
        return height

In [ ]:
def coverage_polygon_to_mask(
    polygon: shapely.geometry.polygon.Polygon,
    center_lon: float,
    center_lat: float,
    extent_lon: float,
    extent_lat: float,
    width: int,
    height: int,
) -> np.ndarray:
    # Calculate region represented by the mask.
    lon_min = center_lon - 0.5 * extent_lon
    lon_max = center_lon + 0.5 * extent_lon

    lat_min = center_lat - 0.5 * extent_lat
    lat_max = center_lat + 0.5 * extent_lat

    # Clip overlay to bbox.
    bbox = gpd.GeoDataFrame(
        geometry=[
            gpd.points_from_xy([lon_min, lon_max], [lat_min, lat_max])
            .union_all()
            .envelope
        ],
        crs="EPSG:4326",
    )
    df = gpd.GeoDataFrame([], geometry=[polygon])
    df.set_crs("EPSG:4326", inplace=True)
    overlay_clipped = df.clip(bbox)

    # Raster transform.
    transform = from_bounds(lon_min, lat_min, lon_max, lat_max, width, height)

    # Rasterize to create binary mask.
    mask = rasterize(
        [(geom, 1) for geom in overlay_clipped.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype=np.uint8,
    )
    return mask

# Config

In [ ]:
radar = Radar(
    id=585,
    lat=47.36700085728634,
    lon=8.537724304199216,
    power=20000,
    diameter=2.0,
    frequency=1,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
)

target_flight_height: float = 1000.0
target_cross_section: float = 2.0

# Parameters for controlling the resolution of the binary mask.
# Mask extent in degrees.
extent = 0.6

# Mask resolution in pixels.
width, height = 1024, 1024

# Run

In [ ]:
client = OpenburstClient("localhost")
coverage = client.calculate_coverage(
    radar,
    target_flight_height,
    target_cross_section,
)

# Plot

In [ ]:
import folium

In [ ]:
m = folium.Map(location=(46.7, 10.3), zoom_start=8)
overlay = folium.GeoJson(coverage).add_to(m)

m.save("map.html")

# Rendering to image

In [ ]:
mask = coverage_polygon_to_mask(
    coverage, radar.lon, radar.lat, extent, extent, width, height
)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(mask, aspect=1.)

# Query elevation

In [ ]:
client.elevation_at(47.38301680962604, 8.495104310332978)